# PySDK training-job status flicker — minimal reproduction

**Context.** In a notebook, when a training job is running, the live status panel *flickers* on every update of the elapsed-time field.

**Real code path.** `trainer` → `AgentRFTJob.wait()` → `sagemaker.train.common_utils.job_wait.wait()` → `_wait_jupyter()`.

**Suspected root cause.** `_wait_jupyter()` renders with a *full clear-and-redraw* cycle every tick:

```python
clear_output(wait=True)      # tear down the whole cell output
console.print(Panel(...))    # reprint the whole panel from scratch
```

Because the panel contains a live `Elapsed Time` field, this fires on every tick, so the entire panel is destroyed and rebuilt — which shows up as flicker.

**This notebook** does NOT need AWS / a real job. It fakes a job whose status/elapsed-time/progress advance over time, and drives the *exact same* rich-panel rendering the real code uses.

## How to use
1. Run the setup cell.
2. Run **Cell A (REPRO)** — this mimics the current `_wait_jupyter` pattern (`clear_output(wait=True)` + `console.print`). Watch for flicker.
3. Run **Cell B (CANDIDATE FIX)** — same content, but repainted in place via `rich.live.Live`. Compare.
4. Run this notebook in **native JupyterLab** and in **Studio UI's JupyterLab** to see whether the flicker is PySDK-side (both flicker) or Studio-frontend-specific (only Studio flickers).


## Setup

In [ ]:
# Only external dep is `rich` (already a PySDK dependency). Uncomment if needed.
# %pip install rich
import rich, IPython
print("rich", rich.__version__)
print("IPython", IPython.__version__)

In [ ]:
import time


class FakeJob:
    """Stand-in for the sagemaker-core Job whose fields advance over time.

    Mirrors just enough of the real Job surface used by the render code:
    job_name, job_arn, job_status, secondary_status, and a training
    progress percentage.
    """

    def __init__(self, total_seconds=40, train_start=6):
        self.job_name = "flicker-repro-training-job"
        self.job_arn = (
            "arn:aws:sagemaker:us-west-2:123456789012:"
            "training-job/flicker-repro-training-job"
        )
        self._start = time.time()
        self._total = total_seconds
        self._train_start = train_start

    @property
    def elapsed(self):
        return time.time() - self._start

    @property
    def job_status(self):
        return "InProgress" if self.elapsed < self._total else "Completed"

    @property
    def secondary_status(self):
        e = self.elapsed
        if e < self._train_start:
            return "Starting"
        if e < self._total:
            return "Training"
        return "Completed"

    @property
    def progress_pct(self):
        e = self.elapsed
        if e < self._train_start:
            return 0.0
        span = max(1e-6, self._total - self._train_start)
        return min(100.0, (e - self._train_start) / span * 100.0)


def build_panel(job):
    """Build the same style of rich Panel the real _wait_jupyter renders.

    Header table + status table (with the live Elapsed Time field) +
    a transitions table containing a training progress bar.
    """
    from rich.console import Group
    from rich.table import Table
    from rich.panel import Panel
    from rich.text import Text
    from rich.box import SIMPLE

    status = job.job_status
    secondary = job.secondary_status
    elapsed = job.elapsed

    # Header
    header = Table(show_header=False, box=None, padding=(0, 1))
    header.add_column("Property", style="cyan bold", width=20)
    header.add_column("Value", style="dim", overflow="fold")
    header.add_row("Job Name", f"[bold green]{job.job_name}[/bold green]")
    header.add_row("Job ARN", f"[dim]{job.job_arn}[/dim]")
    header.add_row("Description", "3 epochs, 100 steps/epoch, batch size 8")

    # Status (Elapsed Time is the field that updates every tick)
    status_t = Table(show_header=False, box=None, padding=(0, 1))
    status_t.add_column("Property", style="cyan bold", width=20)
    status_t.add_column("Value", style="dim")
    status_t.add_row("Job Status", f"[bold][orange3]{status}[/][/]")
    if secondary:
        status_t.add_row("Secondary Status", f"[bold yellow]{secondary}[/bold yellow]")
    status_t.add_row("Elapsed Time", f"[bold bright_red]{elapsed:.1f}s[/bold bright_red]")

    parts = [header, Text(""), status_t]

    # Transitions table with progress bar (only during/after training)
    if secondary in ("Training", "Completed"):
        pct = job.progress_pct
        filled = int(pct / 5)
        bar = (
            f"[green][{'█' * filled}{'░' * (20 - filled)}][/green] "
            f"{pct:.1f}%\n- Step {int(pct)}/100"
        )
        trans = Table(show_header=True, header_style="bold magenta", box=SIMPLE, padding=(0, 1))
        trans.add_column("", style="green", width=2)
        trans.add_column("Step", style="cyan", width=15)
        trans.add_column("Details", style="orange3", width=35)
        trans.add_column("Duration", style="green", width=12)
        trans.add_row("✓", "Starting", "", f"{job._train_start:.1f}s")
        trans.add_row("⋯", "Training", bar, "Running...")
        parts += [Text(""), Text("Status Transitions", style="bold magenta"), trans]

    return Panel(
        Group(*parts),
        title="[bold bright_blue]Agentic RFT Job Status[/bold bright_blue]",
        border_style="orange3",
        width=80,
    )


print("FakeJob and build_panel ready.")

## Cell A — REPRO (current `_wait_jupyter` pattern)

This is the pattern that flickers: `clear_output(wait=True)` then `console.print(...)` on every tick. Renders every 0.5s so the flicker is easy to see. This matches the real code's clear-and-redraw cycle.

In [ ]:
from IPython.display import clear_output
from rich.console import Console

console = Console(force_jupyter=True)
job = FakeJob(total_seconds=40, train_start=6)

while True:
    time.sleep(0.5)
    # --- exact pattern from _wait_jupyter: full clear + full reprint ---
    clear_output(wait=True)
    console.print(build_panel(job))
    # -------------------------------------------------------------------
    if job.job_status == "Completed":
        break

## Cell B — CANDIDATE FIX (`rich.live.Live` repaint-in-place)

Same content, but instead of clearing the whole cell and reprinting, a single `Live` region diffs and rewrites only what changed. Run this in the same environment and compare the visual smoothness against Cell A.

In [ ]:
from rich.live import Live

job = FakeJob(total_seconds=40, train_start=6)

with Live(build_panel(job), refresh_per_second=8, transient=False) as live:
    while True:
        time.sleep(0.5)
        live.update(build_panel(job))
        if job.job_status == "Completed":
            break